# 05e — Comparación de algoritmos de clustering


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering'
REPORT = INFORMES_BASE / '05e_comparison'
REPORT.mkdir(parents=True, exist_ok=True)
VIS = REPORT / 'visualization'
VIS.mkdir(parents=True, exist_ok=True)

dfs = []
for sub, alg in [('05a_kmeans','KMeans'), ('05b_hdbscan','HDBSCAN'), ('05c_gmm','GMM'), ('05d_hierarchical','Hierarchical')]:
    p = INFORMES_BASE / sub / 'metrics.csv'
    if p.exists():
        d = pd.read_csv(p)
        if 'algorithm' not in d.columns:
            d['algorithm'] = alg
        # Normalizar nombre de col K
        if 'min_cluster_size' in d.columns:
            d['k'] = d['n_clusters_actual']
        dfs.append(d)
all_df = pd.concat(dfs, ignore_index=True, sort=False)
all_df.to_csv(REPORT.parent / 'algorithm_comparison.csv', index=False)
print(f'Cargado: {len(all_df)} filas')
print(all_df[['algorithm','version','k','silhouette','davies_bouldin']].to_string(index=False))


Cargado: 93 filas
   algorithm         version  k  silhouette  davies_bouldin
      KMeans v1_conservative  2    0.960110        1.219035
      KMeans v1_conservative  3    0.807673        1.233677
      KMeans v1_conservative  4    0.613709        1.233296
      KMeans v1_conservative  5    0.561921        1.518787
      KMeans v1_conservative  6    0.133927        1.483337
      KMeans v1_conservative  7    0.412303        1.302855
      KMeans v1_conservative  8    0.123854        1.558692
      KMeans v1_conservative  9    0.187029        1.259561
      KMeans v1_conservative 10    0.444307        1.154986
      KMeans v2_intermediate  2    0.960111        1.219035
      KMeans v2_intermediate  3    0.807703        1.233677
      KMeans v2_intermediate  4    0.614402        1.233249
      KMeans v2_intermediate  5    0.567693        1.514113
      KMeans v2_intermediate  6    0.128496        1.489849
      KMeans v2_intermediate  7    0.415531        1.299424
      KMeans v2_interm

In [2]:
# Filtrado viable
viable = all_df.copy()
viable = viable[(viable['n_clusters_actual'] >= 3) & (viable['n_clusters_actual'] <= 10)]
viable = viable[viable['silhouette'] >= 0.15]
if 'outlier_pct' in viable.columns:
    viable = viable[~((viable['algorithm']=='HDBSCAN') & (viable['outlier_pct']>0.30))]

print(f'\nViable: {len(viable)}/{len(all_df)}')
viable['score'] = viable['silhouette'] - 0.1 * viable['davies_bouldin']
viable_sorted = viable.sort_values('score', ascending=False)
print(viable_sorted[['algorithm','version','k','silhouette','davies_bouldin','score']].head(10).to_string(index=False))



Viable: 77/93
   algorithm         version  k  silhouette  davies_bouldin    score
Hierarchical   v3_aggressive  4    0.994877        0.334234 0.961454
Hierarchical v2_intermediate  4    0.994877        0.334234 0.961454
Hierarchical v1_conservative  4    0.994877        0.334234 0.961454
         GMM   v3_aggressive  3    0.993627        0.382765 0.955350
         GMM v2_intermediate  3    0.993627        0.382765 0.955350
         GMM v1_conservative  3    0.993626        0.382765 0.955350
Hierarchical   v3_aggressive  5    0.990852        0.359211 0.954931
Hierarchical v2_intermediate  5    0.990851        0.359211 0.954930
Hierarchical v1_conservative  5    0.990851        0.359211 0.954930
Hierarchical   v3_aggressive  7    0.987195        0.387950 0.948400


In [3]:
# Heatmaps
for metric, fname, cmap in [('silhouette','silhouette_heatmap.png','RdYlGn'),
                             ('davies_bouldin','davies_bouldin_heatmap.png','RdYlGn_r')]:
    pivot = all_df.pivot_table(index=['algorithm','version'], columns='n_clusters_actual', values=metric)
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels([int(c) for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels([f'{a}/{v}' for a,v in pivot.index])
    ax.set_xlabel('n_clusters'); ax.set_title(f'{metric} heatmap')
    for r in range(len(pivot.index)):
        for c in range(len(pivot.columns)):
            v = pivot.iloc[r,c]
            if pd.notna(v): ax.text(c, r, f'{v:.3f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(VIS / fname, dpi=120, bbox_inches='tight')
    plt.close()
print('heatmaps guardados')


heatmaps guardados


In [4]:
# winner_summary.md
if len(viable) > 0:
    winner = viable_sorted.iloc[0]
    status = 'VIABLE'
else:
    winner = all_df.sort_values('silhouette', ascending=False).iloc[0]
    status = 'ESCENARIO ROJO/AMARILLO'

max_sil = all_df['silhouette'].max()
if max_sil >= 0.20:
    scenario = 'VERDE'
elif max_sil >= 0.10:
    scenario = 'AMARILLO'
else:
    scenario = 'ROJO'

lines = [
    f'# Comparación de algoritmos de clustering — gustos',
    '',
    f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Modelos evaluados**: {len(all_df)}',
    f'**Modelos viables (silhouette>=0.15, K en [3,10])**: {len(viable)}',
    f'**Max silhouette observada**: {max_sil:.4f}',
    f'**Escenario**: {scenario}',
    '',
    '## Configuración ganadora',
    '',
    f'- **Algoritmo**: {winner["algorithm"]}',
    f'- **Versión**: {winner["version"]}',
    f'- **K (n_clusters)**: {int(winner["n_clusters_actual"])}',
    f'- **Silhouette**: {winner["silhouette"]:.4f}',
    f'- **Davies-Bouldin**: {winner["davies_bouldin"]:.4f}',
    f'- **Estado**: {status}',
    '',
    '## Top 10 viables',
    '',
    '| Algorithm | Version | K | Silhouette | DB | Score |',
    '|---|---|---:|---:|---:|---:|',
]
if len(viable_sorted) > 0:
    for _, r in viable_sorted.head(10).iterrows():
        lines.append(f"| {r['algorithm']} | {r['version']} | {int(r['n_clusters_actual'])} | {r['silhouette']:.4f} | {r['davies_bouldin']:.4f} | {r['score']:.4f} |")
else:
    lines.append('| _ninguno cumple los criterios_ | | | | | |')

# Interpretación por escenario
lines += ['', '## Interpretación', '']
if scenario == 'VERDE':
    lines.append('Silhouette máximo >= 0.20. Clusters claramente separables. Procede a Fase 5 (caracterización) sin caveats.')
elif scenario == 'AMARILLO':
    lines.append('Silhouette máximo entre 0.10 y 0.20. Clusters parcialmente diferenciados. Caracterización posible pero los arquetipos serán "tendencias" más que "categorías rígidas". Documentar como hallazgo en el TFG.')
else:
    lines.append('Silhouette máximo < 0.10. Los gustos NO se separan en clusters discretos. Hallazgo a defender: el comportamiento de gustos en F2P parece ser esencialmente continuo. Pivotar a PCA + dimensiones de variación.')

(REPORT / 'winner_summary.md').write_text('\n'.join(lines))
print(f'\nwinner_summary.md ({scenario})')
print(f'   Ganador: {winner["algorithm"]} / {winner["version"]} / K={int(winner["n_clusters_actual"])} (sil={winner["silhouette"]:.4f})')



winner_summary.md (VERDE)
   Ganador: Hierarchical / v3_aggressive / K=4 (sil=0.9949)
